# ML System Design — the Senior Round

System design is where ML engineer loops are won at the senior level. The interviewer isn't grading whether you know algorithms — they're watching whether you can **turn a business problem into a system with numbers**: data volumes, latency budgets, training cadence, failure modes. This notebook gives you the 6-step frame, three complete worked designs, and the capacity math executed so your numbers in the interview are practiced, not improvised.

## The 6-Step Frame (Use It Every Time)

1. **Clarify the business problem** — what decision does the model drive? What's the metric *before* ML and what would "good" move it to? Ask about scale, latency, and cost constraints *now*.
2. **Data** — sources, volume, labels (how produced? how delayed?), freshness, quality. Most designs die here — surface label-delay early.
3. **Modeling** — baseline → candidate approaches; batch vs streaming features; the simplest thing that could work, and the upgrade path.
4. **Evaluation** — offline metrics aligned to the decision + online (A/B) plan; guardrail metrics.
5. **Serving & architecture** — batch scoring vs online inference; latency budget; feature store consistency (training/serving skew!).
6. **Monitoring & lifecycle** — drift detection, retraining cadence, rollback, champion/challenger.

**The meta-rule:** spend 5+ minutes on steps 1–2 before saying "model." The candidate who designs the *data flow* first, model second, reads as senior.

## Design 1 — Real-Time Fraud Detection

**Clarify (say this):** card transactions, ~5,000 TPS peak, decision in <100ms inside the authorization path, false positives annoy customers but false negatives cost money directly; labels = chargebacks, **arrive 1–45 days later** (the key constraint).

**Data & features:** transaction stream (Kafka), merchant/customer histories — aggregations like "avg amount last 24h", "distinct merchants this week", "time since last txn", geo-velocity (distance/time between consecutive txns), device fingerprint. Historical features from the feature store; point-in-time correctness so training data matches what serving will see.

**Modeling:** gradient boosting on engineered features (tabular, latency-friendly); sequence model on the card's recent transactions as a stretch. Cost-asymmetric thresholding. **Label delay handling:** train on matured labels only (chargebacks older than 45 days); use delayed-feedback or positive-unlabeled corrections if the business can't wait.

**Serving:** online inference — features precomputed (batch) + on-the-fly (streaming aggregations); model <10ms inference; total budget 100ms. **Fallback:** rule engine (velocity limits) if the model service degrades — never block a payment on ML availability.

**Monitoring:** PSI on features, precision@top-K on matured labels, chargeback rate by segment, latency/error dashboards; champion/challenger with 5% shadow traffic; weekly retrain, instant rollback on drift alarm.

In [1]:
# the capacity math, executed — do this arithmetic out loud in the interview
TPS_peak = 5_000
ms_budget = 100

# feature serving: precomputed vs on-the-fly split
feature_ms = 15        # KV store lookups (Redis/DynamoDB): ~1ms per feature group
model_ms = 8           # GBDT inference on ~200 features
network_ms = 20        # 2 hops
safety_ms = 100 - (feature_ms + model_ms + network_ms)
print(f"latency budget: features {feature_ms}ms + model {model_ms}ms + network {network_ms}ms = "
      f"{feature_ms+model_ms+network_ms}ms, leaving {safety_ms}ms slack (p99 guard)")

txns_day = TPS_peak * 3600 * 24 * 0.3          # 30% duty cycle at peak
labels_day = int(txns_day * 0.002)              # 0.2% chargeback rate
print(f"\ntransactions/day ~{txns_day/1e6:.1f}M; chargeback labels/day ~{labels_day:,}")
print(f"at 0.2% positive rate, a month of matured labels = ~{labels_day*30:,} positives — enough for GBDT,")
print("but only if you WAIT for maturity; training on 7-day labels sees ~40% of the eventual chargebacks")

latency budget: features 15ms + model 8ms + network 20ms = 43ms, leaving 57ms slack (p99 guard)

transactions/day ~129.6M; chargeback labels/day ~259,200
at 0.2% positive rate, a month of matured labels = ~7,776,000 positives — enough for GBDT,
but only if you WAIT for maturity; training on 7-day labels sees ~40% of the eventual chargebacks


## Design 2 — Recommendation System (Feed Ranking)

**Clarify:** item feed ranking, 50M users, 10M items, 200ms p99 end-to-end (includes UI), CTR + session-length + diversity as north-star proxies, A/B-tested against incumbents.

**The two-stage architecture (the expected answer):** retrieval (candidate generation) → ranking. Retrieval: collaborative filtering (matrix factorization / two-tower embeddings) + content-based + trending/popularity — union ~1,000 candidates in <20ms from ANN indexes (HNSW on item/user embeddings). Ranking: heavy GBDT or a small NN on hundreds of features (user-item cross features, freshness, context) scoring the 1,000 → top 50 with diversity re-ranking (MMR-style, category caps).

**Data:** implicit feedback (clicks, dwell, skips — abundant, noisy) + explicit (likes, ratings — sparse, high-signal). **The position-bias question they WILL ask:** clicks correlate with placement — correct with inverse-propensity weighting or a small position-bias model; mention it unprompted for senior points.

**Cold start:** new items → content embeddings + exploration traffic (epsilon-greedy / Thompson sampling bandit); new users → popularity + onboarding preferences.

**Serving:** candidate caches, embedding tables refreshed nightly, ranker retrained daily on yesterday's logs, feature store for user-side aggregates. **Monitoring:** CTR/dwell by segment, novelty/diversity metrics, filter-bubble guardrails, embedding-drift on the two-tower space.

In [2]:
# retrieval funnel math — the numbers to quote
users, items = 50_000_000, 10_000_000
ann_candidates, ranked_output = 1_000, 50

per_user_retrieval_ms = 20
ranker_per_item_ms = 0.15
print(f"retrieval: 10M items -> {ann_candidates} candidates in {per_user_retrieval_ms}ms (ANN, not a scan)")
print(f"ranking: {ann_candidates} items x {ranker_per_item_ms:.2f}ms = {ann_candidates*ranker_per_item_ms:.0f}ms")
print(f"total core logic ~{per_user_retrieval_ms + ann_candidates*ranker_per_item_ms:.0f}ms — fits the 200ms envelope with UI")

daily_impressions = users * 0.4 * 100        # 40% DAU x 100 impressions
print(f"\nimpressions/day ~{daily_impressions/1e9:.1f}B; each one logs features -> ~{daily_impressions*0.3/1e9:.1f}B rows/day")
print("at ~1KB/row that's ~0.3PB/year of training logs — sample or window; don't keep everything forever")

retrieval: 10M items -> 1000 candidates in 20ms (ANN, not a scan)
ranking: 1000 items x 0.15ms = 150ms
total core logic ~170ms — fits the 200ms envelope with UI

impressions/day ~2.0B; each one logs features -> ~0.6B rows/day
at ~1KB/row that's ~0.3PB/year of training logs — sample or window; don't keep everything forever


## Design 3 — Churn Prediction (Batch)

**Clarify:** B2B SaaS, 500K accounts, monthly contract renewals, retention team capacity = 300 outreach/week — the model's job is a **prioritized list**, not probabilities per se. That reframing (ranking + capacity) is the senior move.

**Data:** product telemetry (logins, feature usage trends), support tickets, billing history, contract metadata. Labels = churned/renewed at renewal — **~30–90 day label delay**. **Leakage guards:** everything as-of the scoring date only (no post-hoc features like "days since last login" computed today for a renewal three months ago).

**Modeling:** gradient boosting with trend features (usage slope over 8 weeks vs previous 8), account-health aggregates, ticket sentiment. Calibrated probabilities → expected-value ranking: P(churn) × contract value = expected loss — the retention team works the top of *that* list, not the top of P(churn) (small accounts churn at 90% and it still doesn't matter — say this and watch the interviewer nod).

**Serving:** batch weekly scoring (no latency pressure), output to CRM with reasons (top SHAP values per account — actionability). **Evaluation:** offline AUC/PR + calibration; online = controlled experiment on outreach (treated vs held-out churn comparison, or uplift modeling if the outreach itself has heterogeneous effects). **Monitoring:** score distribution drift, realized churn vs predicted (calibration over time), feature freshness from upstream pipelines.

In [3]:
# the expected-value ranking, executed — why it beats raw P(churn)
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
accounts = pd.DataFrame({
    "account": [f"A{i}" for i in range(12)],
    "p_churn": np.round(rng.beta(2, 6, 12), 2),
    "annual_value": rng.choice([2_000, 15_000, 120_000], 12),
})
accounts["expected_loss"] = accounts.p_churn * accounts.annual_value

print(accounts.sort_values("p_churn", ascending=False).head(6)[["account", "p_churn", "annual_value", "expected_loss"]].to_string(index=False))
print()
top_p = accounts.sort_values("p_churn", ascending=False).account.iloc[:3].tolist()
top_ev = accounts.sort_values("expected_loss", ascending=False).account.iloc[:3].tolist()
print(f"top-3 by P(churn):        {top_p}   <- small accounts")
print(f"top-3 by expected loss:   {top_ev}   <- the accounts that actually matter")

account  p_churn  annual_value  expected_loss
     A6     0.43          2000          860.0
     A5     0.26        120000        31200.0
     A7     0.24          2000          480.0
     A0     0.20          2000          400.0
     A8     0.19        120000        22800.0
    A11     0.17          2000          340.0

top-3 by P(churn):        ['A6', 'A5', 'A7']   <- small accounts
top-3 by expected loss:   ['A5', 'A8', 'A9']   <- the accounts that actually matter


## The Cross-Cutting Questions (Asked in Every Design Round)

**Training/serving skew** — features computed differently offline vs online (batch SQL vs streaming code paths). Fix: one definition, two runtimes via a feature store, or compute-on-write parity tests.

**Batch vs online inference** — batch: cheap, stale (hours), fine for churn/credit; online: fresh, latency/infra cost, required for fraud/search. Say the tradeoff explicitly, pick per problem.

**Retraining cadence** — driven by drift rate vs label delay vs cost; not "weekly" as dogma. Champion/challenger + canary promotion beats scheduled redeploys.

**Feedback loops** — the model changes user behavior, which changes tomorrow's training data (recommendations especially). Mitigate: exploration traffic, held-out control users, off-policy correction (inverse propensity weighting).

**Privacy/regulatory** — PII minimization, consent, right-to-deletion (features derived from deleted users must be purgeable — feature stores need lineage for this), model explainability requirements (credit/fraud: adverse-action reasons).

**Cost** — training compute vs incremental metric gain; inference $/1M requests; the interviewer loves a candidate who closes with "and here's what it costs."**

In [4]:
# the drift-retraining trigger math — a concrete policy to quote
monthly_drift = 0.02           # PSI units per month on key features
alert_threshold = 0.10         # act at PSI > 0.10
retrain_interval_months = alert_threshold / monthly_drift
print(f"at ~{monthly_drift} PSI/month drift, degradation hits the alert line in ~{retrain_interval_months:.0f} months")
print("so: monthly retrain is comfortable; quarterly is riding the edge; annual is negligence")
print("pair the schedule with event triggers: sudden PSI jump or metric drop = retrain NOW, not next Tuesday")

at ~0.02 PSI/month drift, degradation hits the alert line in ~5 months
so: monthly retrain is comfortable; quarterly is riding the edge; annual is negligence
pair the schedule with event triggers: sudden PSI jump or metric drop = retrain NOW, not next Tuesday


## Rehearsal Protocol for Design Rounds

1. **Whiteboard the 6-step frame first** — literally write the six headers; it structures your 45 minutes and shows method.
2. **Do the arithmetic out loud** — TPS, storage/day, labels/day, latency split. Numbers are the senior signal; vagueness is the junior one.
3. **Ask before assuming** — scale, latency, cost, evaluation: the questions ARE the design.
4. **Name the failure modes unprompted** — skew, drift, feedback loops, cold start, fallbacks. Each one you raise is a point the interviewer doesn't have to probe.
5. **Close with tradeoffs and costs** — "I'd ship the GBDT first for latency, keep the sequence model as challenger; here's the monthly infra bill" — decisions, not hedging.

The three designs above (fraud, recsys, churn) cover ~80% of design-round prompts; the LLM/RAG variant — increasingly the 2026 default — is the [next notebook](LLM%20and%20RAG%20System%20Design.ipynb).